In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import sys
import os
import pickle

from scipy.signal import find_peaks
from scipy.optimize import curve_fit

current_dir = '/home/marian/CIGAR_ANALYSIS/CIGAR/notebooks'

# Build the absolute path to ../functions
functions_path = os.path.abspath(os.path.join(current_dir, '../functions'))

# Add it to sys.path
sys.path.append(functions_path)

import parse_data
import cigar as cig

from tqdm import tqdm

In [2]:
colors = ['royalblue', 'crimson', 'green', 'darkorange', 'brown', 'blue', 'coral', 'indigo', 'magenta', 'black']
CB_MARKERS = ['o', 's', '^', 'D', 'v', 'P', '*', 'X']

font_size = 20
plt.rcParams['font.family'] = 'serif'
plt.rcParams['mathtext.fontset'] = 'cm'
plt.rcParams['font.size'] = font_size


##  Runs

Inputs: one {pressure: run_base} dict per gas — the dedicated per-channel SPhe calibration runs used in `FingerPlot4CHs.ipynb` (each run lives in 4 directories, `run_base + '1'` .. `run_base + '4'`, one channel per .bin file), not the alpha/signal runs `ChargeSpectrum.ipynb` reads. DCR is computed from a charge finger plot of these calibration runs (multi-Gaussian peak fit + a Poisson-family fit on the peak areas), the same method as `FingerPlot4CHs.ipynb`, instead of from the noise tail of the physics runs.

The values below are copied from `FingerPlot4CHs.ipynb` - only `Ar @ 8.5bar` (`Run172SPheCH`) and `Xe @ 1.5bar` (`Run115SPheCH`) have actual calibration data on disk right now; the rest are still placeholders (`Run0SPheCH`) to fill in as more calibration runs are taken.

In [ ]:
base = "/home/marian/CIGAR_ANALYSIS/CIGAR/data"

# {pressure [bar]: run_base} per gas - the dedicated per-channel SPhe
# calibration runs used by FingerPlot4CHs.ipynb (each run lives in 4
# directories, run_base + '1'..'4', one channel per .bin file), NOT the
# alpha/signal runs ChargeSpectrum.ipynb reads.
runs = {
    'Ar': {8.5: f'Run{172}SPheCH'
           ,7.5: f'Run{00}SPheCH'
           ,6.5: f'Run{00}SPheCH'
           ,5.5: f'Run{00}SPheCH'
           ,4.5: f'Run{00}SPheCH'
           ,3.5: f'Run{00}SPheCH'
           ,2.5: f'Run{00}SPheCH'
           ,1.5: f'Run{00}SPheCH'
           },
    'Xe': {8.5: f'Run{00}SPheCH'
           ,7.5: f'Run{00}SPheCH'
           ,6.5: f'Run{00}SPheCH'
           ,5.5: f'Run{00}SPheCH'
           ,4.5: f'Run{00}SPheCH'
           ,3.5: f'Run{00}SPheCH'
           ,2.5: f'Run{00}SPheCH'
           ,1.5: f'Run{115}SPheCH'
           },
}

# temp labels - same convention as FingerPlot4CHs.ipynb / ChargeSpectrum.ipynb
temperatures = {
    'Ar': {8.5: f'{6}deg', 7.5: f'{7}deg', 6.5: f'{8}deg', 5.5: f'{9}deg',
           4.5: f'{10}deg', 3.5: f'{11}deg', 2.5: f'{12}deg', 1.5: f'{13.5}deg'},
    'Xe': {8.5: f'{8}deg', 7.5: f'{10}deg', 6.5: f'{12}deg', 5.5: f'{13}deg',
           4.5: f'{11.5}deg', 3.5: f'{13}v2deg', 2.5: f'{13}v3deg', 1.5: f'{14}deg'},
}

# DAQ parameters for the calibration runs (see FingerPlot4CHs.ipynb) - one
# channel per .bin file (4 directories per run), unlike the 10-channel
# packed format of the alpha/signal runs
channels             = ['CH1', 'CH2', 'CH3', 'CH4']
nchannels            = 1        # channels packed into each .bin file
nevents_per_wvf      = 2000
samples_per_waveform = 124
event_header_bytes   = 28
sample_binning       = 8e-9     # seconds/sample

start  = 1
nfiles = None   # None = read every file in the run directory; lower this for a quick test

# Number of Gaussians to fit in the finger plot, per (gas, pressure) - see
# FingerPlot4CHs.ipynb. Only the runs actually calibrated so far have a
# tuned value; any other (gas, pressure) falls back to a generic guess that
# will likely need adjusting once that run's data is fit for the first time.
n_gaussians_default = [4, 4, 4, 4]
n_gaussians = {
    'Ar': {8.5: [5, 5, 3, 4]},
    'Xe': {},
}

##  Reading + dark count rate computation

In [ ]:
def load_run(run_base, base_dir, channels, nchannels, nevents_per_wvf,
             event_header_bytes, start=0, nfiles=None, sample_binning=8e-9):
    """
    Reads one calibration run into a single DataFrame - each channel lives
    in its own run directory (run_base + '1', ..., run_base + '4'), same
    layout as FingerPlot4CHs.ipynb, unlike the alpha/signal runs (all
    channels packed into one .bin file per event).
    """
    channel_dfs = {}
    for ch_idx, ch in enumerate(channels):
        channel_num = ch_idx + 1
        run         = f'{run_base}{channel_num}'
        run_dir     = f'{base_dir}/{run}'

        dataframes = []
        for i, file in enumerate(tqdm(os.listdir(run_dir)[start:nfiles],
                                       desc=f'Reading {ch} .bin files', unit='file')):
            if file.endswith('.bin'):
                file_path = os.path.join(run_dir, file)
                df_tmp = parse_data.parse_wf_from_binary(
                    file_path,
                    channels=nchannels,
                    n_events=nevents_per_wvf,
                    file_idx=i,
                    event_header_bytes=event_header_bytes,
                    sample_binning=sample_binning
                )
                dataframes.append(df_tmp)

        ch_df = pd.concat(dataframes, ignore_index=True)
        ch_df.rename(columns={'CH1': ch}, inplace=True)   # parse_data always returns 'CH1'
        channel_dfs[ch] = ch_df

    # Merge all channels on event + TIME (they share the same event structure)
    df = channel_dfs[channels[0]][['event', 'TIME', channels[0]]].copy()
    for ch in channels[1:]:
        df = df.merge(channel_dfs[ch][['event', 'TIME', ch]], on=['event', 'TIME'], how='inner')

    return df

In [ ]:
def compute_charge(df, channels, nevents, time, time_window, baselines):
    """
    Per-channel per-waveform integrated charge [mV*s] over the fixed
    charge-integration window - same as FingerPlot4CHs.ipynb. No amp-factor
    rescaling or p.e. calibration here: this is the raw charge the
    finger-plot Gaussians (FingerPlot below) are fit to directly, and N_pe
    comes from the fitted peak index, not from a separate calibration
    constant.
    """
    charge = {}
    for ch in channels:
        voltage            = df[ch].values
        v_matrix           = voltage.reshape(nevents, -1)
        v_matrix_corrected = v_matrix - baselines[ch]
        charge[ch] = np.trapz(v_matrix_corrected[:, time_window], x=time[time_window], axis=1)
    return charge


def FingerPlot(charge, channels, n_gaussians_list, gas, pressure, run_base, plot=True):
    """
    Charge finger plot for all channels in a 2x2 grid - same method as
    FingerPlot4CHs.ipynb: histogram each channel's integrated charge, find
    the N_pe peaks, and fit a sum of Gaussians to get each peak's
    height/mean/width.
    """
    opt_A, opt_mu, opt_sigma, opt_mu_errors = {}, {}, {}, {}

    if plot:
        fig, axs = plt.subplots(2, 2, figsize=(20, 16), dpi=150)
        fig.suptitle(f'{run_base}s ({gas} @ {pressure}bar) — Charge Finger Plot')
        axs_flat = axs.flatten()

    for i, ch in enumerate(channels):
        n_gaussians = n_gaussians_list[i]
        vals        = charge[ch]

        if i < 2:
            hist_range = (0, 3.e-4)
        elif i == 2:
            hist_range = (0, 2.5e-4)
        else:
            hist_range = (0.5e-4, 3.5e-4)

        if plot:
            ax = axs_flat[i]
            events_hist, bins, _ = ax.hist(vals, bins=int(np.sqrt(len(vals))), range=hist_range,
                                           density=False, color=colors[i], alpha=0.3, label=ch)
        else:
            events_hist, bins = np.histogram(vals, bins=int(np.sqrt(len(vals))), range=hist_range)

        bins_means    = (bins[:-1] + bins[1:]) / 2
        peak_distance = 40 if i < 2 else 60
        peaks, _      = find_peaks(events_hist, distance=peak_distance, prominence=0.6)
        if plot:
            ax.plot(bins_means[peaks], events_hist[peaks], 'or', markersize=20, alpha=0.3)

        msk = ((bins_means > (bins_means[events_hist.argmax()] + bins_means[0]) * 0.05) &
               (bins_means < bins_means[peaks[n_gaussians - 1]] * 1.15))
        x = bins_means[msk]
        y = events_hist[msk]
        initial_guess_A     = list(events_hist[peaks][:n_gaussians])
        initial_guess_mu    = list(bins_means[peaks][:n_gaussians])
        initial_guess_sigma = [0.6 * (bins_means[1] - bins_means[0])] * n_gaussians
        initial_guess       = initial_guess_A + initial_guess_mu + initial_guess_sigma
        params, covariance  = curve_fit(cig.sum_of_gaussians, x, y, p0=initial_guess, maxfev=100000)
        for _ in range(10):
            lower  = np.minimum(params * 0.8, params * 1.2)
            upper  = np.maximum(params * 0.8, params * 1.2)
            params, covariance = curve_fit(cig.sum_of_gaussians, x, y, p0=params,
                                            bounds=(lower, upper), maxfev=100000)

        opt_A[ch]         = params[:n_gaussians]
        opt_mu[ch]        = params[n_gaussians:2*n_gaussians]
        opt_sigma[ch]     = params[2*n_gaussians:]
        opt_mu_errors[ch] = np.sqrt(np.diag(covariance)[n_gaussians:2*n_gaussians])

        if plot:
            gaussian_sum = cig.sum_of_gaussians(x, *params)
            ax.plot(x, gaussian_sum, color=colors[i], lw=3, alpha=1, label='Gaussian Fit')

            for jj, mu in enumerate(opt_mu[ch]):
                y_marker = cig.sum_of_gaussians(mu, *params)
                one_g    = [opt_A[ch][jj], opt_mu[ch][jj], opt_sigma[ch][jj]]
                ax.plot(mu, y_marker, 'X', color='k', markersize=10, zorder=5)
                ax.plot(x, cig.sum_of_gaussians(x, *one_g), '--', color='k')
                ax.annotate(fr'$\mu_{{{jj+1}}}$={mu:.2E}', xy=(mu, y_marker), xytext=(0, 12),
                            textcoords='offset points', ha='center', va='bottom',
                            fontsize=font_size * 0.8, color='k')

            ax.xaxis.set_major_formatter(plt.matplotlib.ticker.ScalarFormatter(useMathText=True))
            ax.ticklabel_format(axis='x', style='sci', scilimits=(0, 0))
            ax.set_title(ch)
            ax.set_xlabel(r'Charge [mV·s]')
            ax.set_ylabel('Counts')
            ax.grid(True)
            ax.legend(fontsize=font_size)

    if plot:
        plt.tight_layout()
        plt.show()

    return opt_A, opt_mu, opt_sigma, opt_mu_errors


def drop_pedestal(opt_A, opt_mu, opt_sigma, opt_mu_errors):
    """
    The Gaussians in opt_* are ordered by increasing charge (peak 1 =
    lowest). If peak 1 is shorter than peak 2, it's the pedestal (0 p.e.),
    not the 1-p.e. peak - drop it so the peaks fed into DCR below start at
    N_pe = 1, otherwise the Poisson fit would be shifted by one peak and
    the DCR would come out wrong (see FingerPlot4CHs.ipynb).
    """
    for ch in opt_A:
        if opt_A[ch][0] < opt_A[ch][1]:
            print(f'{ch}: peak 1 is shorter than peak 2 -> treating peak 1 as the pedestal (0 p.e.) and dropping it')
            opt_A[ch]         = opt_A[ch][1:]
            opt_mu[ch]        = opt_mu[ch][1:]
            opt_sigma[ch]     = opt_sigma[ch][1:]
            opt_mu_errors[ch] = opt_mu_errors[ch][1:]
    return opt_A, opt_mu, opt_sigma, opt_mu_errors

In [ ]:
def calculate_DCR(opt_A, opt_mu, opt_sigma, window_width_time, gas=None, pressure=None, run_base=None, plot=True):
    """
    Fit each channel's charge-peak areas (N_pe = 1, 2, 3, ...) to a
    Generalized Poisson / Borel-Poisson distribution (Consul & Jain 1973;
    Vinogradov 2012 for the SiPM application) instead of a plain Poisson:
    a Poisson(theta) number of primary dark pulses, each independently
    spawning a Borel(lam)-distributed crosstalk/afterpulse cascade. lam = 0
    recovers plain Poisson (FingerPlot4CHs.ipynb's calculate_DCR). mu =
    theta / (1 - lam) is the mean number of p.e. per integration window;
    dividing by the window width [s] gives the DCR [Hz]. lam is the fitted
    crosstalk/afterpulsing probability (0-1).

    Each Gaussian from FingerPlot is unnormalized (A = peak height, not
    area), so the "number of counts" under peak j is its integral
    A_j * sigma_j * sqrt(2*pi), not A_j alone.
    """
    DCR, DCR_err, lam_fit, lam_err, poisson_params = {}, {}, {}, {}, {}

    if plot:
        fig, axs = plt.subplots(2, 2, figsize=(18, 12), dpi=150, constrained_layout=True)
        title = 'Dark Count Rate'
        if gas is not None:
            title += f' — {gas} @ {pressure}bar'
        if run_base:
            title += f' ({run_base}s)'
        fig.suptitle(title)
        axs_flat = axs.flatten()

    for idx, ch in enumerate(opt_A.keys()):
        # Order peaks by charge (Npe = 1, 2, 3, ...) and turn each Gaussian
        # into its area (= total counts under that peak), not just its height.
        order  = np.argsort(opt_mu[ch])
        npe    = np.arange(len(order)) + 1
        counts = (opt_A[ch] * opt_sigma[ch] * np.sqrt(2 * np.pi))[order]

        theta0 = np.sum(npe * counts) / np.sum(counts)
        p0     = [counts.max(), theta0, 0.1]
        bounds = ([0, 1e-6, 0], [np.inf, np.inf, 0.95])   # A >= 0, theta > 0, 0 <= lam < 1
        params, covariance = curve_fit(cig.generalized_poisson_continuous, npe, counts,
                                        p0=p0, bounds=bounds, maxfev=100000)
        A_fit, theta, lam = params

        mu         = theta / (1 - lam)
        dmu_dtheta = 1 / (1 - lam)
        dmu_dlam   = theta / (1 - lam) ** 2
        var_mu     = (dmu_dtheta ** 2 * covariance[1, 1] + dmu_dlam ** 2 * covariance[2, 2]
                      + 2 * dmu_dtheta * dmu_dlam * covariance[1, 2])
        mu_err = np.sqrt(max(var_mu, 0))

        DCR[ch]            = mu / window_width_time
        DCR_err[ch]        = mu_err / window_width_time
        lam_fit[ch]        = lam
        lam_err[ch]        = np.sqrt(covariance[2, 2])
        poisson_params[ch] = params

        if plot:
            ax = axs_flat[idx]
            npe_fit = np.linspace(1, npe.max(), 200)
            ax.plot(npe, counts, 'o', color='k', markersize=12, label='Peak area (counts)')
            ax.plot(npe_fit, cig.generalized_poisson_continuous(npe_fit, *params), color=colors[idx], lw=3,
                    label=fr'$\mu$={mu:.2f}$\pm${mu_err:.2f}, $\lambda$={lam*100:.1f}%')

            ax.yaxis.set_major_formatter(plt.matplotlib.ticker.ScalarFormatter(useMathText=True))
            ax.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))

            DCR_MHz     = DCR[ch] / 1e6
            DCR_err_MHz = DCR_err[ch] / 1e6
            ax.text(0.4, 0.3, f'DCR = ({DCR_MHz:.2f} ± {DCR_err_MHz:.2f}) MHz',
                    transform=ax.transAxes, ha='right', va='top', fontsize=font_size * 0.8,
                    bbox=dict(boxstyle='round', facecolor='white', edgecolor='k', alpha=0.8))

            ax.set_title(ch, fontsize=font_size)
            ax.set_xlabel(r'$N_{pe}$', fontsize=font_size)
            ax.set_ylabel('Counts (peak area)', fontsize=font_size)
            ax.grid(True)
            ax.legend(fontsize=font_size * 0.7, loc='upper right')

    if plot:
        plt.show()

    return DCR, DCR_err, lam_fit, lam_err, poisson_params

##  Step-by-step example: one run

Pick one `(gas, pressure)` pair and walk through the full pipeline - waveform density, integration window, the charge finger plot, and the DCR it implies - before running every pressure at the end. `gas` chosen here also drives the "all pressures" pass at the end of the notebook; it is not looped over.

In [ ]:
# --- everything below (including the "all pressures" pass at the end) uses this gas ---
gas = 'Ar'   # 'Xe' or 'Ar'

# which single run to walk through step by step first
example_pressure = 8.5   # [bar]

In [ ]:
run_base = runs[gas][example_pressure]

df_example = load_run(run_base, base, channels, nchannels, nevents_per_wvf,
                       event_header_bytes, start=start, nfiles=nfiles,
                       sample_binning=sample_binning)

event_list = df_example['event'].unique()
nevents    = len(event_list)
print(f'{gas} @ {example_pressure} bar ({run_base}s): {nevents} events')

###  Baseline

Same method as `ChargeSpectrum.ipynb`'s / `FingerPlot4CHs.ipynb`'s baseline-correction step (histogram the pretrigger samples, locate the peak with mode + local quadratic-in-log fit), computed from this run's pretrigger samples on all 4 channels - check the fitted red curve visually tracks the histogram peak.

In [ ]:
def compute_baselines(df, nevents, pretrigger, channels, gas=None, pressure=None, run_base=None, plot=True):
    """
    Per-channel baseline offset from the pretrigger samples - same method
    as ChargeSpectrum.ipynb's / FingerPlot4CHs.ipynb's "Baseline
    correction" step: histogram the first `pretrigger` samples of every
    waveform (no real signal should have arrived yet), then locate the
    peak by fitting a quadratic to log(counts + 1) in a small window
    around the tallest bin (equivalent to a local Gaussian fit, since the
    log of a Gaussian is a parabola, but more robust to noisy bins in the
    wings than fitting the raw counts directly). Falls back to the raw
    histogram mode if that fit fails.

    Returns a {'CH1': ..., 'CH2': ..., 'CH3': ..., 'CH4': ...} dict, keyed
    by channel name, as expected by compute_charge/FingerPlot below.
    """
    if plot:
        fig, axs = plt.subplots(2, 2, figsize=(16, 10), dpi=150)
        title = f'Baseline (pretrigger={pretrigger})'
        if gas is not None:
            title += f' — {gas} @ {pressure} bar'
        if run_base:
            title += f' ({run_base}s)'
        fig.suptitle(title)
        axs_flat = axs.flatten()

    fixBaselines = {}
    for i, ch in enumerate(channels):
        channel = i + 1

        voltage   = df[ch].values
        v_matrix  = voltage.reshape(nevents, -1)
        baselines = v_matrix[:, :pretrigger].flatten()

        counts, bins = np.histogram(baselines, bins=int(0.3 * np.sqrt(len(baselines))),
                                     range=(baselines.min(), baselines.max()))
        bin_centers  = 0.5 * (bins[:-1] + bins[1:])
        mode_initial = bin_centers[np.argmax(counts)]

        bin_width = bins[1] - bins[0]
        window    = 0.05 * len(bins) * bin_width
        fit_mask  = (bin_centers >= mode_initial - window) & (bin_centers <= mode_initial + window)
        x_fit = bin_centers[fit_mask]
        y_fit = np.log(counts[fit_mask] + 1)

        coeffs = None
        try:
            coeffs = np.polyfit(x_fit, y_fit, 2)
            a, b, _ = coeffs
            peak_value = -b / (2 * a)
        except Exception:
            print(f'Baseline fit failed for {ch}, using histogram mode instead')
            peak_value = mode_initial

        fixBaselines[ch] = peak_value

        if plot:
            ax = axs_flat[i]
            ax.hist(baselines, bins=bins, color=colors[i], alpha=0.7, label=ch)
            if coeffs is not None:
                x_fine = np.linspace(x_fit.min(), x_fit.max(), 200)
                ax.plot(x_fine, np.exp(np.polyval(coeffs, x_fine)) - 1, '-r', lw=2, label='Quadratic fit')
            ax.axvline(peak_value, color='red', linestyle='--', lw=2, label=f'{peak_value:.1f} mV')
            ax.set_xlabel('Baseline [mV]')
            ax.set_ylabel('Counts')
            ax.set_yscale('log')
            ax.grid(True)
            ax.legend(fontsize=font_size * 0.7)

    if plot:
        plt.tight_layout()
        plt.show()

    return fixBaselines

In [ ]:
pretrigger = 12   # pretrigger samples used as the baseline region - waveform is only 124 samples long (see FingerPlot4CHs.ipynb)

fixBaselines = compute_baselines(df_example, nevents, pretrigger, channels, gas, example_pressure, run_base, plot=True)
print('fixBaselines:', fixBaselines)

##  Waveform density + integration window

The charge-integration window is the fixed window from `FingerPlot4CHs.ipynb` (0.18-0.6 µs after trigger) - there's no separate "signal window" to exclude here, since these are dedicated SPhe calibration acquisitions, not alpha/physics runs. It may need retuning per run/pressure if the SPhe pulse timing shifts.

In [ ]:
t_matrix = df_example['TIME'].values.reshape(nevents, -1)
time     = t_matrix[0]

# Charge integration window for the finger plot - same as FingerPlot4CHs.ipynb.
time_window = (0.18e-6 < time) & (time <= 0.6e-6)

In [ ]:
def plot_waveform_density(df, nevents, channels, baselines, time, time_window, gas, pressure, run_base):
    """
    2x2 grid, one hist2d per channel, showing where every waveform in the
    run sits in (time, ADC) space. The shaded band marks the charge
    integration window used by the finger plot below - same window as
    FingerPlot4CHs.ipynb.
    """
    fig, axs = plt.subplots(2, 2, figsize=(20, 12), dpi=150, sharex=True, sharey=False)
    fig.suptitle(f'Waveform density — {gas} @ {pressure} bar ({run_base}s)')
    axs_flat = axs.flatten()

    t_matrix = df['TIME'].values.reshape(nevents, -1)

    for i, ch in enumerate(channels):
        ax = axs_flat[i]

        voltage            = df[ch].values
        v_matrix           = voltage.reshape(nevents, -1)
        v_matrix_corrected = v_matrix - baselines[ch]

        hb = ax.hist2d(t_matrix.flatten(), v_matrix_corrected.flatten(),
                       bins=[np.shape(t_matrix)[1], 200],
                       cmap='plasma', norm=mcolors.PowerNorm(gamma=0.3))
        cbar = fig.colorbar(hb[3], ax=ax, label='Counts')
        cbar.formatter.set_powerlimits((0, 0))
        cbar.formatter.set_useMathText(True)

        ax.fill_between(time[time_window], v_matrix_corrected.min(), v_matrix_corrected.max(),
                        color='white', alpha=0.25, label='Integration window (charge)')

        ax.set_title(ch)
        ax.set_ylabel('ADC [mV]')
        ax.set_xlabel('Time (s)')
        ax.grid(True)
        ax.legend(fontsize=font_size * 0.6, framealpha=0.5)

    plt.tight_layout()
    plt.show()
    return fig, axs

In [ ]:
fig, axs = plot_waveform_density(df_example, nevents, channels, fixBaselines, time, time_window, gas, example_pressure, run_base)

###  Charge Finger Plot

N_pe here comes directly from the fitted peak index (1, 2, 3, ...), not from a separate p.e. calibration constant - see `FingerPlot4CHs.ipynb`.

In [ ]:
n_gaussians_list = n_gaussians.get(gas, {}).get(example_pressure, n_gaussians_default)

charge_example = compute_charge(df_example, channels, nevents, time, time_window, fixBaselines)

opt_A, opt_mu, opt_sigma, opt_mu_errors = FingerPlot(charge_example, channels, n_gaussians_list,
                                                      gas, example_pressure, run_base)

opt_A, opt_mu, opt_sigma, opt_mu_errors = drop_pedestal(opt_A, opt_mu, opt_sigma, opt_mu_errors)

###  Dark Count Rate

In [ ]:
window_width_time = time[time_window][-1] - time[time_window][0]

DCR, DCR_err, lam_fit, lam_err, poisson_params = calculate_DCR(opt_A, opt_mu, opt_sigma, window_width_time,
                                                                 gas, example_pressure, run_base)

for ch in DCR:
    print(f'{ch}: DCR = ({DCR[ch]:.3E} ± {DCR_err[ch]:.1E}) Hz, '
          f'crosstalk lambda = ({lam_fit[ch]*100:.1f} ± {lam_err[ch]*100:.1f})%  '
          f'(integration window = {window_width_time:.2E} s)')

##  All pressures

The step above walked through one run; now repeat the same pipeline for every pressure of the chosen `gas`. Reading all files (`nfiles = None`) for 8 runs × 4 channels can take a while - lower `nfiles` above while testing.

In [ ]:
all_pressures = False
if all_pressures:
    dcr_results = {}

    for pressure, run_base_p in tqdm(runs[gas].items(), desc=f'{gas} pressures'):
        df_p = load_run(run_base_p, base, channels, nchannels, nevents_per_wvf,
                        event_header_bytes, start=start, nfiles=nfiles,
                        sample_binning=sample_binning)

        event_list_p = df_p['event'].unique()
        nevents_p    = len(event_list_p)

        t_matrix_p    = df_p['TIME'].values.reshape(nevents_p, -1)
        time_p        = t_matrix_p[0]
        time_window_p = (0.18e-6 < time_p) & (time_p <= 0.6e-6)

        # recompute the baseline for this run - it's a property of each run, not a global constant
        fixBaselines_p = compute_baselines(df_p, nevents_p, pretrigger, channels, gas, pressure, run_base_p, plot=True)

        n_gaussians_list_p = n_gaussians.get(gas, {}).get(pressure, n_gaussians_default)
        charge_p            = compute_charge(df_p, channels, nevents_p, time_p, time_window_p, fixBaselines_p)
        opt_A_p, opt_mu_p, opt_sigma_p, opt_mu_errors_p = FingerPlot(charge_p, channels, n_gaussians_list_p,
                                                                      gas, pressure, run_base_p)
        opt_A_p, opt_mu_p, opt_sigma_p, opt_mu_errors_p = drop_pedestal(opt_A_p, opt_mu_p, opt_sigma_p, opt_mu_errors_p)

        window_width_time_p = time_p[time_window_p][-1] - time_p[time_window_p][0]
        DCR_p, DCR_err_p, lam_fit_p, lam_err_p, poisson_params_p = calculate_DCR(
            opt_A_p, opt_mu_p, opt_sigma_p, window_width_time_p, gas, pressure, run_base_p)

        dcr_results[pressure] = {
            ch: dict(DCR=DCR_p[ch], DCR_err=DCR_err_p[ch], lam=lam_fit_p[ch], lam_err=lam_err_p[ch])
            for ch in DCR_p
        }

##  Dark count rate vs pressure

In [ ]:
def plot_dcr_vs_pressure(dcr_results, gas, channels):
    fig, ax = plt.subplots(1, 1, figsize=(10, 7), dpi=150)
    ax.set_title(f'Dark count rate vs pressure — {gas}')

    pressures_sorted = sorted(dcr_results.keys())

    for i, ch in enumerate(channels):
        dcr_vals = [dcr_results[p][ch]['DCR']     / 1e6 for p in pressures_sorted]
        dcr_errs = [dcr_results[p][ch]['DCR_err'] / 1e6 for p in pressures_sorted]

        ax.errorbar(pressures_sorted, dcr_vals, yerr=dcr_errs,
                    fmt=CB_MARKERS[i], color=colors[i], label=ch,
                    markersize=8, capsize=4)

    ax.set_xlabel('Pressure [bar]')
    ax.set_ylabel('Dark count rate [MHz]')
    ax.grid(True)
    ax.legend()
    plt.tight_layout()
    plt.show()
    return fig, ax

In [ ]:
if all_pressures:
    fig, ax = plot_dcr_vs_pressure(dcr_results, gas, channels)

##  Save results

In [21]:
save_results = False
if (save_results) & (all_pressures):
    out_dir = f'{base}/dcr'
    os.makedirs(out_dir, exist_ok=True)
    with open(f'{out_dir}/dcr_results_{gas}.pkl', 'wb') as f:
        pickle.dump(dcr_results, f)
    print(f'Saved DCR results to {out_dir}/dcr_results_{gas}.pkl')